# 01 — Residual Momentum Baseline

**Role**: Comparison baseline against which the H6 Sub-Industry Leader-Follower strategy will be evaluated.

## Methodology

Within-Sub-Industry residual momentum is the published-literature workhorse for clean cross-sectional momentum: Blitz, Huij & Martens (2011) demonstrate that the residual from a CAPM-style factor regression generates a Sharpe ratio materially higher than raw return momentum, with substantially lower drawdowns. The mechanism is that raw momentum partly captures sector cycles and thus inherits sector-rotation risk; residualization isolates the genuinely idiosyncratic component.

This notebook implements the strategy end-to-end and reports gross and net performance. It is not intended to be the production alpha — it is the rigorous baseline against which any novel signal (here, the H6 leader-follower) must demonstrate incremental value.

### Six-step pipeline

1. **Map** each name to its GICS-Sector SPDR ETF (XLK, XLF, XLE, …).
2. **Estimate** rolling beta of name return against matched ETF return (126-day window).
3. **Compute** residual return = name_return − beta × etf_return.
4. **Aggregate** the signal as cumulative residual over a 6-month lookback, skipping the most recent month (avoids 1-month reversal).
5. **Rank** cross-sectionally within Sub-Industry.
6. **Construct** dollar-neutral long-short portfolio: top quintile long, bottom quintile short, equal-weighted within side.

### Design choices

- **Beta lookback = 126 days** (≈ 6 months). Sensitivity tested in §10.
- **Momentum lookback = 126 days, skip = 21 days**. Standard skip-month momentum construction.
- **Quintile portfolio**, not deciles. Quintiles maintain breadth (≈ 20% of universe long, 20% short), reducing idiosyncratic noise while preserving signal.
- **Equal-weighted within bucket**. Cap-weighted would tilt the portfolio toward mega-caps and dilute the signal; equal weighting is more honest for a baseline.
- **One-day holding**. Each day's weights earn the next-day return. Strategy turnover is therefore high; expect cost drag to dominate gross-to-net.
- **Dollar-neutral by construction**. Sum of weights equals zero per date; long side totals +1, short side −1, gross 2.0.

### What this baseline shows / does not show

The baseline establishes the expected return per unit of risk available from the standard residual momentum factor in this universe. It does not validate the H6 hypothesis; that work is in notebook 03. Performance differences between H6 and this baseline measure whether the leader-follower mechanism captures alpha incremental to known residual momentum.


---
## 0. Setup


In [ ]:
from __future__ import annotations
import warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from lsa.data import (
    load_pit_panel, clean_pit_panel, merge_pit_panels, INDEX_FILES,
)
from lsa.signals import (
    sector_to_etf_mapping,
    build_residual_momentum_strategy,
)

warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING, format='%(levelname)s [%(name)s] %(message)s')
plt.rcParams.update({
    'figure.figsize': (11, 5), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})
NAVY, GOLD, MAROON, GREEN = '#1F3A5F', '#B8862E', '#8E2F2F', '#1B5E20'

DATA_DIR = Path('../data')
print(f'Data directory: {DATA_DIR.resolve()}')


---
## 1. Load & clean panels
Loads SP400/500/600, runs the cleaner pipeline (gap-aware returns, winsorization), merges. This is the same data layer used by every notebook in the project.


In [ ]:
panels = {}
for idx, (fn, _) in INDEX_FILES.items():
    raw = load_pit_panel(DATA_DIR / fn, idx)
    panel, _ = clean_pit_panel(raw)
    panels[idx] = panel
merged = merge_pit_panels(panels)
print(f'Merged panel: {len(merged):,} rows, {merged["ID"].nunique():,} unique IDs, '
      f'{merged["DATE"].min().date()} → {merged["DATE"].max().date()}')


---
## 2. Build ETF return panel
Wide form (date × ETF ticker) of daily ETF returns. We use the unadjusted `Close.pct_change()` to match the PIT panel's price-return convention (PIT Price is split-adjusted but not dividend-adjusted).


In [ ]:
etf = pd.read_csv(DATA_DIR / 'etf_ohlcv_20160301_20251231.csv', parse_dates=['Date'])
etf_wide_close = etf.pivot(index='Date', columns='Ticker', values='Close').sort_index()
etf_returns_wide = etf_wide_close.pct_change()
print(f'ETF returns shape: {etf_returns_wide.shape}; tickers: {sorted(etf_returns_wide.columns.tolist())}')

# Confirm sector-ETF coverage
sec2etf = sector_to_etf_mapping()
missing = [e for e in sec2etf.values() if e not in etf_returns_wide.columns]
assert not missing, f'Missing ETFs: {missing}'
print('Sector ETF mapping fully resolved.')


---
## 3. Run the strategy end-to-end
Single call to the orchestrator. Returns the panel with all intermediate columns (`beta`, `residual`, `signal`, `rank_pct`, `weight`), the daily strategy return series, and a diagnostics dict.


In [ ]:
result = build_residual_momentum_strategy(
    merged,
    etf_returns_wide,
    window=126,             # rolling beta lookback
    min_periods=63,
    lookback_days=126,      # 6-month momentum window
    skip_days=21,           # skip last month
    top_pct=0.20,           # top quintile long
    bottom_pct=0.20,        # bottom quintile short
    min_cohort_size=4,      # sub-industries with < 4 names dropped
    holding_days=1,         # daily rebalance, 1-day hold
)

print('Diagnostics:', result.diagnostics)
print(f'Strategy return series: {len(result.daily_returns):,} days, '
      f'first non-NaN: {result.daily_returns.first_valid_index()}, '
      f'last: {result.daily_returns.last_valid_index()}')


**Dollar neutrality check.** The diagnostics dict reports max-absolute net-weight per date and the count of dates that violate the 1e-6 tolerance. Both should be effectively zero — the construction enforces neutrality by design, so any drift would signal a numerical bug.


---
## 4. Beta diagnostics
Beta stability is a precondition for clean residualization. Two views: cross-sectional distribution at a representative date, and one name's beta time series.


In [ ]:
betas = result.betas.dropna()
recent = betas[betas['DATE'] == betas['DATE'].max()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(recent['beta'], bins=40, color=NAVY, alpha=0.85)
axes[0].axvline(1.0, color=MAROON, linestyle='--', alpha=0.7, label='β = 1')
axes[0].set_title(f'Cross-section of beta on {recent["DATE"].iloc[0].date()}')
axes[0].set_xlabel('β to sector ETF (126-day rolling)')
axes[0].set_ylabel('Names')
axes[0].legend()

sample_id = 'AAPL UW Equity'
sample_b = betas[betas['ID'] == sample_id].set_index('DATE')['beta']
if len(sample_b) > 0:
    sample_b.plot(ax=axes[1], color=NAVY, lw=1.4)
    axes[1].axhline(1.0, color=MAROON, linestyle='--', alpha=0.5)
    axes[1].set_title(f'{sample_id} rolling beta to XLK')
    axes[1].set_xlabel('Date'); axes[1].set_ylabel('β')
plt.tight_layout(); plt.show()


Beta cross-section centers near 1 with a moderate spread — names are partly exposed to their sector and partly idiosyncratic. The AAPL beta time series should be stable around 1.0–1.2 with limited regime drift; large swings would indicate either a benchmark-fit problem or a real change in the name's sector exposure.


---
## 5. Residual return diagnostics
After residualization, the residual should have substantially lower variance than the raw return (sector beta removed) and near-zero serial correlation. We verify both.


In [ ]:
panel = result.panel.dropna(subset=['residual']).copy()
sample_ids = panel.groupby('ID').size().sort_values(ascending=False).head(50).index
sub = panel[panel['ID'].isin(sample_ids)]

raw_std = sub.groupby('ID')['ret'].std() * 100
res_std = sub.groupby('ID')['residual'].std() * 100
var_red = 1 - (res_std / raw_std) ** 2  # variance reduction fraction

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(raw_std, bins=25, color=GOLD, alpha=0.6, label='raw return std')
axes[0].hist(res_std, bins=25, color=NAVY, alpha=0.7, label='residual std')
axes[0].set_title('Std distribution: raw vs residual (% daily)')
axes[0].set_xlabel('Daily std (%)'); axes[0].set_ylabel('Names'); axes[0].legend()

axes[1].hist(var_red.dropna(), bins=25, color=GREEN, alpha=0.85)
axes[1].axvline(var_red.median(), color='black', linestyle='--',
                label=f'median = {var_red.median()*100:.1f}%')
axes[1].set_title('Variance reduction from residualization')
axes[1].set_xlabel('1 − Var(residual) / Var(raw)')
axes[1].set_ylabel('Names'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Median variance reduction: {var_red.median()*100:.1f}%')


Median variance reduction in the 30–50% range is typical and confirms the sector beta is doing material work. Names with low reduction (< 10%) are effectively sector-independent; names with reduction > 70% are highly sector-driven and have very small residuals — both extremes are valid but at the extremes the signal becomes noisier per unit volatility.


---
## 6. Signal distribution and rank check


In [ ]:
sig = result.panel.dropna(subset=['signal'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(sig['signal'].clip(-0.5, 0.5), bins=80, color=NAVY, alpha=0.85)
axes[0].set_title('Residual momentum signal distribution\n(126d cumulative residual, skip 21d)')
axes[0].set_xlabel('Signal'); axes[0].set_ylabel('Observations')

# Rank consistency check
ranks = sig.dropna(subset=['rank_pct'])
axes[1].hist(ranks['rank_pct'], bins=20, color=GOLD, alpha=0.85)
axes[1].set_title('Within-Sub-Industry rank distribution\n(should be approximately uniform on [0, 1])')
axes[1].set_xlabel('Rank percentile'); axes[1].set_ylabel('Observations')
plt.tight_layout(); plt.show()


Signal histogram should be approximately symmetric around zero with fat tails (most names have small cumulative residuals; a minority show pronounced over/underperformance). The rank histogram must be flat — that confirms the cross-sectional ranking machinery is working without ties or distribution artifacts.


---
## 7. Strategy performance
Cumulative equity curve, rolling 252-day Sharpe, drawdown. Gross only at this stage — costs added in §8.


In [ ]:
daily = result.daily_returns.dropna()
equity = (1 + daily).cumprod()
drawdown = equity / equity.cummax() - 1
rolling_sharpe = (daily.rolling(252).mean() / daily.rolling(252).std()) * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
equity.plot(ax=axes[0], color=NAVY, lw=1.6)
axes[0].set_title('Cumulative equity (gross, dollar-neutral, 2.0 gross exposure)')
axes[0].set_ylabel('Equity')
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.2f}x'))

drawdown.plot(ax=axes[1], color=MAROON, lw=1.2)
axes[1].fill_between(drawdown.index, drawdown.values, 0, color=MAROON, alpha=0.20)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown')
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))

rolling_sharpe.plot(ax=axes[2], color=GREEN, lw=1.4)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].axhline(rolling_sharpe.mean(), color=NAVY, linestyle='--',
                label=f'full-sample Sharpe = {(daily.mean()/daily.std()*np.sqrt(252)):.2f}')
axes[2].set_title('Rolling 252-day Sharpe')
axes[2].set_ylabel('Sharpe'); axes[2].set_xlabel('Date'); axes[2].legend()
plt.tight_layout(); plt.show()


---
## 8. Performance statistics


In [ ]:
def perf_stats(r: pd.Series, freq: int = 252) -> dict:
    r = r.dropna()
    mu, sd = r.mean(), r.std()
    sharpe = mu / sd * np.sqrt(freq)
    equity = (1 + r).cumprod()
    dd = (equity / equity.cummax() - 1).min()
    cagr = equity.iloc[-1] ** (freq / len(r)) - 1
    hit = (r > 0).mean()
    return {
        'n_days':       len(r),
        'CAGR':         cagr,
        'ann_return':   mu * freq,
        'ann_vol':      sd * np.sqrt(freq),
        'Sharpe':       sharpe,
        'max_DD':       dd,
        'hit_rate':     hit,
        'skew':         r.skew(),
        'kurtosis':     r.kurtosis(),
    }

stats = perf_stats(result.daily_returns)
for k, v in stats.items():
    if k == 'n_days':
        print(f'  {k:<15}{v}')
    elif k in ('CAGR', 'ann_return', 'ann_vol', 'max_DD', 'hit_rate'):
        print(f'  {k:<15}{v*100:>+8.2f}%')
    else:
        print(f'  {k:<15}{v:>+8.3f}')


---
## 9. Turnover and cost sensitivity
At daily rebalance, turnover is high (≈ 2000-3000% annualized). Cost drag is the dominant factor between gross IR and net IR.


In [ ]:
def daily_turnover(panel: pd.DataFrame, id_col='ID', date_col='DATE', wt_col='weight') -> pd.Series:
    wide = panel.pivot_table(index=date_col, columns=id_col, values=wt_col, fill_value=0.0).sort_index()
    return wide.diff().abs().sum(axis=1) / 2.0  # one-way turnover, since |Δw|/2 = trades

trnvr = daily_turnover(result.panel.dropna(subset=['rank_pct']))
ann_turnover = trnvr.mean() * 252
print(f'Daily one-way turnover (mean): {trnvr.mean()*100:.1f}% of gross book')
print(f'Annualized turnover:           {ann_turnover*100:.0f}%')

# Apply a flat cost assumption per single side, then sweep
daily = result.daily_returns
cost_grid_bps = [0, 5, 10, 15, 20, 30]
cost_panel = []
for c in cost_grid_bps:
    cost_drag = trnvr * (c / 10_000) * 2     # round-trip = 2x one-way
    net = (daily - cost_drag).dropna()
    s = net.mean() / net.std() * np.sqrt(252)
    cost_panel.append({'cost_per_side_bps': c, 'net_Sharpe': s})
cost_df = pd.DataFrame(cost_panel)
cost_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(cost_df['cost_per_side_bps'], cost_df['net_Sharpe'], marker='o', color=NAVY, lw=1.5)
ax.axhline(0, color=MAROON, linestyle='--', alpha=0.5, label='Sharpe = 0')
ax.set_title('Net Sharpe sensitivity to transaction cost')
ax.set_xlabel('Per-side cost (bps)'); ax.set_ylabel('Net Sharpe')
ax.legend()
plt.tight_layout(); plt.show()


The break-even cost (intersection with Sharpe = 0) tells us the cost ceiling at which the strategy stops paying. For a daily-rebalance baseline, break-even is typically 10-20 bps per side. Realistic blended costs at SP400/600 are well above 10 bps per side, so this baseline is essentially break-even after costs in production — exactly the property we want from a benchmark.


---
## 10. Sensitivity sweeps (sanity checks)
Two cheap robustness tests: (a) parameter sensitivity on the momentum lookback, (b) sub-period stability.


In [ ]:
lookbacks = [63, 126, 252]
skips     = [5, 21]
rows = []
for lb in lookbacks:
    for sk in skips:
        if sk >= lb: continue
        r = build_residual_momentum_strategy(
            merged, etf_returns_wide,
            window=126, min_periods=63,
            lookback_days=lb, skip_days=sk,
            top_pct=0.20, bottom_pct=0.20,
            min_cohort_size=4, holding_days=1,
        ).daily_returns.dropna()
        rows.append({
            'lookback_days': lb,
            'skip_days':     sk,
            'gross_Sharpe':  r.mean() / r.std() * np.sqrt(252),
            'ann_return':    r.mean() * 252,
            'ann_vol':       r.std() * np.sqrt(252),
        })
sens = pd.DataFrame(rows)
sens


In [ ]:
# Sub-period stability
d = result.daily_returns.dropna()
splits = [('2016-2019', d.loc[:'2019-12-31']),
          ('2020-2025', d.loc['2020-01-01':])]
for name, sub in splits:
    s = sub.mean()/sub.std()*np.sqrt(252)
    print(f'{name}: n_days={len(sub):>5}, ann_return={sub.mean()*252*100:>+7.2f}%, '
          f'ann_vol={sub.std()*np.sqrt(252)*100:>5.2f}%, Sharpe={s:>+5.2f}')


Sub-period IRs should be within ~1-2 standard errors of each other for the strategy to be considered stable. A clear regime break (e.g., positive 2016-2019, negative 2020-2025) would invalidate the baseline.


---
## 11. Conclusions and limitations

### What this baseline establishes

- A reproducible, dollar-neutral residual-momentum strategy on the S&P 1500 universe with gross Sharpe in the historically expected 0.5–0.9 range.
- A break-even cost level that the H6 leader-follower must improve upon to justify research investment.
- Beta-stability and variance-reduction diagnostics that any subsequent residualization-based signal can reuse.

### Honest limitations

1. **Cost-fragile**. The daily rebalance schedule and high turnover make net performance dominated by the cost model. The H6 leader-follower is also high-turnover, so the cost ceiling computed here transfers — but the sensitivity is real.
2. **Sector-ETF residualization is imperfect for SP400/600**. The SPDR sector ETFs hold only SP500 names; using them as the benchmark for SP400/600 leaves a small/mid-cap idiosyncratic component in the residual. This is acceptable for a baseline (and we trade that residual deliberately) but is worth flagging.
3. **Pre-2018 Communication Services**. Pre-XLC-inception (2018-06-19), Communication Services names produce NaN betas and are excluded. This biases the universe before mid-2018.
4. **No earnings calendar**. Standard practice in residual momentum is to exclude announcement windows; this baseline does not, because the dataset lacks announcement dates. The contribution of earnings-window observations is therefore not isolated.

### Next step

Open **notebook 03_leadlag_validation.ipynb** to run the H6 hypothesis test. The H6 strategy will be evaluated against this baseline's net Sharpe directly.
